# Which data did the metaheuristics search in the submitted run?

Manuscript CMPBUP-D-25-00249R1. Section 2.6.1 (lines 432–436) says that the training set was used for feature
selection and that the hold-out test set was *"strictly isolated and unseen during all phases of model development"*.
This notebook checks which file the reducer (DE, GA and SA) searched in the submitted run. The explanation and what it
means for the manuscript are in [README.md](README.md).

**Run all cells from the top.** Section 1 lists every file the notebook reads; nothing is written to disk.
Sections 2–4 and 6 need only Python. Section 5 needs Weka, which needs a Java runtime (`JAVA_HOME` in section 1).

| Section | Question |
|---|---|
| 2 | Is `normal_rtf.csv` the training rows plus the test rows? |
| 3 | Which file does the old pipeline load for the metaheuristics? |
| 4 | Are the reducer's own output files the subsets reported in the manuscript? |
| 5 | On which file do the reported merits, and the DE subsets, come out? |
| 6 | How were the χ² merits of Table 2 normalized? |
| 7 | Summary of all checks |

Genes are 0-based column indices, as in the result files. Gene *g*N in the manuscript is column N − 1.

## 1. Setup and files

In [1]:
import ast
import json
import logging
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# All paths are relative to the repository root. The notebook lives in audit/reducer_input/:
# VS Code gives its path in __vsc_ipynb_file__, and plain Jupyter starts the kernel in the notebook's folder.
if '__vsc_ipynb_file__' in globals():
    os.chdir(Path(__vsc_ipynb_file__).resolve().parents[2])
elif Path.cwd().name == 'reducer_input':
    os.chdir(Path.cwd().parents[1])
assert Path('ofunctions.py').is_file(), f'Start the kernel in the repository root: the working directory is {os.getcwd()}'
sys.path.insert(0, os.getcwd())   # to import ofunctions and baselines from the repository

pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

In [2]:
# ---- Every file this notebook reads (paths relative to the repository root). Nothing is written. ----

# Data, in data/, min-max scaled to [0, 1]
FULL_FILE = 'normal_rtf.csv'                        # 396 rows: the whole balanced data set
TRAIN_FILE = 'train_normal_rtf.csv'                 # 356 rows: the training partition
TEST_FILE = 'test_normal_rtf.csv'                   # 40 rows: the submitted test set (30 real patients, 10 SMOTE rows)
REAL_TEST_FILE = 'test_rtf_v2_cleaned_normal.csv'   # the 30 real test patients

# Subsets and merits reported in the manuscript, one spreadsheet per round (Tables 2 and 3)
SUBMITTED = {r: f'experiment_results/subset_r{r}.xlsx' for r in (1, 2, 3)}

# Unmodified copies of files of the old project, /mnt/Cris-SW/Research/MetaheuristicCollection (see README.md)
EVIDENCE = 'audit/reducer_input/evidence'
PIPELINES = [f'{EVIDENCE}/pipeline.py', f'{EVIDENCE}/pipeline_2.py', f'{EVIDENCE}/pipeline_3.py']
REDUCER_OUTPUTS = {r: f'{EVIDENCE}/minimum_length_resume_step_{r}.csv' for r in (1, 2, 3)}   # written by the reducer
NORMALIZATION_NOTEBOOK = f'{EVIDENCE}/processDataset.ipynb'

# Values printed in Table 3 of the manuscript
TABLE3_CHI2 = {'DE': 37.181, 'GA': 34.505, 'SA': 32.255}   # aCHI2 merit of each metaheuristic's subset
# Column "Whole set": Eq. (11) merit of the 70 genes. aMI is printed as 0.034 for SA and 0.021 for DE and GA
TABLE3_WHOLE_SET = {'mi': 0.034, 'sim_uncertainty': 0.031, 'gain_ratio': 0.037, 'relieff': 0.010, 'chi2': 14.573}

# Objectives whose per-gene score does not depend on the other genes: under |S| >= 10, the 10 best-ranked genes
# maximize Eq. (11), so a search that reaches the optimum returns exactly that top 10
SEPARABLE_OBJECTIVES = ['mi', 'sim_uncertainty', 'gain_ratio', 'chi2', 'information_gain']
K = 10

# Java runtime for Weka (section 5). There is no system Java on this machine; any Java 11 or later works.
# A JAVA_HOME already set in the environment takes precedence.
JAVA_HOME = '/opt/JetBrains Rider-2026.1.2/jbr'

# Two merits are equal if they differ by less than this (the result files keep 7 or 8 significant digits)
TOLERANCE = 1e-6

In [3]:
def load(name):
    # Genes and labels of a file in data/
    data = pd.read_csv(f'data/{name}')
    return data.drop(columns=['label']).to_numpy(), data['label'].to_numpy(dtype=np.intp)


def read_results(path):
    # Best subset per metaheuristic and objective: a spreadsheet of the manuscript or an output file of the reducer.
    # Both have the columns Model, Iteration, Subset, Length, Name, Settings, Merit
    if path.endswith('.xlsx'):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, encoding='utf-8-sig')   # the reducer's files start with a byte-order mark
    return pd.DataFrame({'Meta': df['Model'], 'OF': df['Name'], 'Length': df['Length'],
                         'Subset': df['Subset'].map(ast.literal_eval), 'Merit': df['Merit'].astype(float)})


def same(a, b, tolerance=TOLERANCE):
    return abs(a - b) <= tolerance


CHECKS = []


def statement(text, holds):
    # Records one checked statement; section 7 lists them all
    CHECKS.append({'Statement': text, 'Holds': bool(holds)})
    print(('HOLDS          ' if holds else 'DOES NOT HOLD  ') + text)

## 2. The data files

A row is compared by its label and its 70 genes rounded to 9 decimals (the scaled files differ by about 1e-16).

In [4]:
X_full, y_full = load(FULL_FILE)
X_train, y_train = load(TRAIN_FILE)
X_test, y_test = load(TEST_FILE)
X_real, y_real = load(REAL_TEST_FILE)


def rows(X, y):
    return {(int(label), *np.round(genes, 9)) for genes, label in zip(X, y)}


full, train, test, real = rows(X_full, y_full), rows(X_train, y_train), rows(X_test, y_test), rows(X_real, y_real)
display(pd.DataFrame({
    'file': [FULL_FILE, TRAIN_FILE, TEST_FILE, REAL_TEST_FILE],
    'rows': [len(y_full), len(y_train), len(y_test), len(y_real)],
    'distinct rows': [len(full), len(train), len(test), len(real)],
    f'of them in {FULL_FILE}': [len(full), len(train & full), len(test & full), len(real & full)],
}))
statement(f'{FULL_FILE} is exactly the rows of {TRAIN_FILE} plus those of {TEST_FILE}, and no row is in both',
          full == (train | test) and not (train & test))
statement(f'The 30 real test patients ({REAL_TEST_FILE}) are among the 40 rows of {TEST_FILE}', real <= test)
statement(f'The 30 real test patients are rows of {FULL_FILE}', real <= full)

,file,rows,distinct rows,of them in normal_rtf.csv
0,normal_rtf.csv,396,396,396
1,train_normal_rtf.csv,356,356,356
2,test_normal_rtf.csv,40,40,40
3,test_rtf_v2_cleaned_normal.csv,30,30,30


HOLDS          normal_rtf.csv is exactly the rows of train_normal_rtf.csv plus those of test_normal_rtf.csv, and no row is in both
HOLDS          The 30 real test patients (test_rtf_v2_cleaned_normal.csv) are among the 40 rows of test_normal_rtf.csv
HOLDS          The 30 real test patients are rows of normal_rtf.csv


## 3. Which file the old pipeline loads

The old pipeline is an interactive menu (`main_option`). For each of its three copies, the cell lists:

* every call to `processDataSet`, with the menu option it belongs to and the file it loads;
* every menu option that runs metaheuristics, and the data it gives them. Two options run the multi-round reducer:
  "Cargar y correr configuración" (6) and "Correr paso desde archivo de resumen" (9 in `pipeline.py`, 8 in the other
  two copies, whose menus have no MLP option). `getMetasByObjectiveFunction` creates DE, GA and SA on the data it
  receives, and the rounds are written to `step_N_report.csv`. Options 4 and 5 run single experiments
  (`runExperiment`) on a data set picked by hand in option 1, and write no round reports.

Each copy's menu labels are read from that copy.

**Caveat.** `pipeline.py` and `pipeline_2.py` were last modified on 16 March 2026 and `pipeline_3.py` on
23 April 2025, after the run (September 2023). So they may not be exactly the code that ran. Sections 4–6 do not
depend on the code: they use the run's output files and the data.

In [5]:
OPTION = re.compile(r'(?:el)?if main_option == (\d+)')
LITERAL = re.compile(r"processDataSet\('([^']+)'\)")
NAME_FILE = re.compile(r"NAME_FILE = '([^']+)'")


def data_file(line, block_text):
    # File loaded by one processDataSet call: a literal name, NAME_FILE as set in the same block, or a manual choice
    literal = LITERAL.search(line)
    if literal:
        return literal.group(1)
    if 'NAME_FILE' in line:
        name = NAME_FILE.search(block_text)
        return f'{name.group(1)}.csv' if name else 'NAME_FILE'
    return 'chosen by hand in option 1'


loads, runs, comments = [], [], []
for path in PIPELINES:
    name = Path(path).name
    lines = Path(path).read_text(encoding='utf-8', errors='replace').splitlines()
    blocks, option, menu = {}, None, {}
    for number, line in enumerate(lines, start=1):
        match = OPTION.search(line)
        if match:
            option = int(match.group(1))
        if option is not None:
            blocks.setdefault(option, []).append((number, line))
        if not menu and 'Menu principal' in line:
            menu = {int(k): v for k, v in re.findall(r'(\d)\. ([^\\\'"]+)', line)}
        if 'generated with all data' in line:
            comments.append({'file': name, 'line': number, 'menu option': option, 'comment': line.strip()})
    for option, block in blocks.items():
        text = '\n'.join(line for _, line in block)
        block_loads = [(number, line) for number, line in block if 'processDataSet(' in line]
        for number, line in block_loads:
            loads.append({'file': name, 'line': number, 'menu option': option, 'menu label': menu.get(option),
                          'data file': data_file(line, text), 'code': line.strip()})
        if 'getMetasByObjectiveFunction(' in text or 'runExperiment(' in text:
            runs.append({'file': name, 'menu option': option, 'menu label': menu.get(option),
                         'writes the round reports (step_N_report.csv)': 'step_{i + 1}_report.csv' in text,
                         'data given to the metaheuristics': ', '.join(sorted({data_file(line, text) for _, line in block_loads}))
                                                             or 'the data set chosen by hand in option 1'})

loads, runs = pd.DataFrame(loads), pd.DataFrame(runs)
print('Every data load in a menu option:')
display(loads)
print('Menu options that run metaheuristics:')
display(runs)
print("The comment next to the reducer's rounds:")
display(pd.DataFrame(comments))

reducer = runs[runs['writes the round reports (step_N_report.csv)']]
options_by_file = reducer.groupby('file')['menu option'].apply(lambda o: ' and '.join(str(i) for i in sorted(o)))
statement(f'In the three copies, every menu option that runs the multi-round reducer loads {FULL_FILE} ('
          + ', '.join(f'options {o} in {f}' for f, o in options_by_file.items()) + ')',
          len(reducer) > 0 and (reducer['data given to the metaheuristics'] == FULL_FILE).all())
mh_options = set(zip(runs['file'], runs['menu option']))
train_loads = loads[loads['data file'] == TRAIN_FILE]
statement(f'{TRAIN_FILE} is loaded only by menu options that do not run metaheuristics',
          not any((f, o) in mh_options for f, o in zip(train_loads['file'], train_loads['menu option'])))

Every data load in a menu option:


,file,line,menu option,menu label,data file,code
0,pipeline.py,484,1,Establecer dataset,chosen by hand in option 1,"data, labels = processDataSet(data_set)"
1,pipeline.py,679,6,Cargar y correr configuración,normal_rtf.csv,"data, labels = processDataSet('normal_rtf.csv')"
2,pipeline.py,815,7,Ejecutar pipeline de clasificadores a tabla de resumen,train_normal_rtf.csv,"data, labels = processDataSet(f'{NAME_FILE}.csv')"
3,pipeline.py,864,8,Ejecutar evaluación con MLP,train_normal_rtf.csv,"data, labels = processDataSet(f'{NAME_FILE}.csv')"
4,pipeline.py,926,9,Correr paso desde archivo de resumen,normal_rtf.csv,"data, labels = processDataSet('normal_rtf.csv')"
5,pipeline_2.py,484,1,Establecer dataset,chosen by hand in option 1,"data, labels = processDataSet(data_set)"
6,pipeline_2.py,679,6,Cargar y correr configuración,normal_rtf.csv,"data, labels = processDataSet('normal_rtf.csv')"
7,pipeline_2.py,814,7,Ejecutar pipeline de clasificadores a tabla de resumen,train_normal_rtf.csv,"data, labels = processDataSet(f'{NAME_FILE}.csv')"
8,pipeline_2.py,872,8,Correr paso desde archivo de resumen,normal_rtf.csv,"data, labels = processDataSet('normal_rtf.csv')"
9,pipeline_3.py,485,1,Establecer dataset,chosen by hand in option 1,"data, labels = processDataSet(data_set)"


Menu options that run metaheuristics:


,file,menu option,menu label,writes the round reports (step_N_report.csv),data given to the metaheuristics
0,pipeline.py,4,Ejecutar algoritmos seleccionados,False,the data set chosen by hand in option 1
1,pipeline.py,5,Ejecutar modo wrapper,False,the data set chosen by hand in option 1
2,pipeline.py,6,Cargar y correr configuración,True,normal_rtf.csv
3,pipeline.py,9,Correr paso desde archivo de resumen,True,normal_rtf.csv
4,pipeline_2.py,4,Ejecutar algoritmos seleccionados,False,the data set chosen by hand in option 1
5,pipeline_2.py,5,Ejecutar modo wrapper,False,the data set chosen by hand in option 1
6,pipeline_2.py,6,Cargar y correr configuración,True,normal_rtf.csv
7,pipeline_2.py,8,Correr paso desde archivo de resumen,True,normal_rtf.csv
8,pipeline_3.py,4,Ejecutar algoritmos seleccionados,False,the data set chosen by hand in option 1
9,pipeline_3.py,5,Ejecutar modo wrapper,False,the data set chosen by hand in option 1


The comment next to the reducer's rounds:


,file,line,menu option,comment
0,pipeline.py,700,6,#Here mh_dict is generated with all data
1,pipeline.py,952,9,#Here mh_dict is generated with all data
2,pipeline_2.py,700,6,#Here mh_dict is generated with all data
3,pipeline_2.py,898,8,#Here mh_dict is generated with all data
4,pipeline_3.py,701,6,#Here mh_dict is generated with all data
5,pipeline_3.py,899,8,#Here mh_dict is generated with all data


HOLDS          In the three copies, every menu option that runs the multi-round reducer loads normal_rtf.csv (options 6 and 9 in pipeline.py, options 6 and 8 in pipeline_2.py, options 6 and 8 in pipeline_3.py)
HOLDS          train_normal_rtf.csv is loaded only by menu options that do not run metaheuristics


## 4. The reducer's output files and the manuscript

`minimum_length_resume_step_{1,2,3}.csv` (dated 21 September 2023) are the reducer's results of each round: the
best subset per metaheuristic and objective, with the merit computed during the search. `subset_r{1,2,3}.xlsx`
hold the values reported in Tables 2 and 3.

In [6]:
summary, round3 = [], None
for r in (1, 2, 3):
    reducer = read_results(REDUCER_OUTPUTS[r]).set_index(['Meta', 'OF'])
    submitted = read_results(SUBMITTED[r]).set_index(['Meta', 'OF'])
    joined = reducer.merge(submitted, left_index=True, right_index=True, how='outer', suffixes=('_reducer', '_manuscript'))
    joined['same subset'] = [sorted(a) == sorted(b) for a, b in zip(joined['Subset_reducer'], joined['Subset_manuscript'])]
    joined['same merit'] = [same(a, b) for a, b in zip(joined['Merit_reducer'], joined['Merit_manuscript'])]
    not_chi2 = joined.index.get_level_values('OF') != 'chi2'
    summary.append({'round': r, 'subsets': len(joined), 'same subset': int(joined['same subset'].sum()),
                    'same merit (not χ²)': f"{int(joined.loc[not_chi2, 'same merit'].sum())} of {int(not_chi2.sum())}",
                    'same merit (χ²)': f"{int(joined.loc[~not_chi2, 'same merit'].sum())} of {int((~not_chi2).sum())}"})
    if r == 3:
        round3 = joined
display(pd.DataFrame(summary))
print('Round 3:')
display(round3[['Length_reducer', 'Merit_reducer', 'Merit_manuscript', 'same subset', 'same merit']])
statement('The reducer output files hold the same subsets as the manuscript spreadsheets, in all three rounds',
          all(s['same subset'] == s['subsets'] for s in summary))
statement('Their merits are also the same, except for χ², which the spreadsheets hold normalized (section 6)',
          all(s['same merit (not χ²)'].split(' of ')[0] == s['same merit (not χ²)'].split(' of ')[1] for s in summary))

,round,subsets,same subset,same merit (not χ²),same merit (χ²)
0,1,18,18,15 of 15,0 of 3
1,2,18,18,15 of 15,0 of 3
2,3,18,18,15 of 15,0 of 3


Round 3:


Length_reducer  Merit_reducer  Merit_manuscript  same subset  same merit
Meta OF                                                                                        
DE   chi2                          10      37.181303          1.000000         True       False
     gain_ratio                    10       0.103818          0.103818         True        True
     information_gain              10       0.071213          0.071213         True        True
     mi                            10       0.082621          0.082621         True        True
     relieff                       18       0.014495          0.014495         True        True
     sim_uncertainty               10       0.078548          0.078548         True        True
GA   chi2                          10      34.505093          0.879358         True       False
     gain_ratio                    10       0.090621          0.090621         True        True
     information_gain              10       0.064532          0.064532         True        True
     mi                            10       0.078270          0.078270         True        True
     relieff                       11       0.013328          0.013328         True        True
     sim_uncertainty               10       0.071600          0.071600         True        True
SA   chi2                          10      32.255296          0.777938         True       False
     gain_ratio                    10       0.078635          0.078635         True        True
     information_gain              10       0.056368          0.056368         True        True
     mi                            10       0.067145          0.067145         True        True
     relieff                       10       0.012647          0.012647         True        True
     sim_uncertainty               10       0.060747          0.060747         True        True

HOLDS          The reducer output files hold the same subsets as the manuscript spreadsheets, in all three rounds
HOLDS          Their merits are also the same, except for χ², which the spreadsheets hold normalized (section 6)


## 5. Recomputing the merits and the top-10 ranking on each file

With the repository's code (`ofunctions.allObjectiveFunctionWeka`, which the metaheuristics optimize, and
`baselines.top_k_filter.rank_features`):

* **5.1** the Eq. (11) merit of every subset of the three rounds, on the training file and on the full file,
  against the merit the reducer logged during the search;
* **5.2** the merit of the 70 genes, against the column "Whole set" of Table 3;
* **5.3** the 10 best-ranked genes of each objective, against the DE subsets.

The Weka JVM can be started only once per kernel: to run the next cell again, restart the kernel. When it starts, the
JVM prints warnings about native access; they do not affect the results.

In [7]:
os.environ.setdefault('JAVA_HOME', JAVA_HOME)
import weka.core.jvm as jvm
import ofunctions
from baselines.top_k_filter import rank_features

FILES = {TRAIN_FILE: (X_train, y_train), FULL_FILE: (X_full, y_full)}
reducer_rounds = {r: read_results(REDUCER_OUTPUTS[r]) for r in (1, 2, 3)}

merits, whole_set, rankings = [], [], {}
jvm.start(packages=True, logging_level=logging.WARNING)
try:
    for r, results in reducer_rounds.items():
        for _, row in results.iterrows():
            values = {name: ofunctions.allObjectiveFunctionWeka(row['Subset'], X, y, type_of=row['OF'])
                      for name, (X, y) in FILES.items()}
            merits.append({'round': r, 'Meta': row['Meta'], 'OF': row['OF'], 'L': row['Length'],
                           'logged by the reducer': row['Merit'], **values})
    for of in TABLE3_WHOLE_SET:
        values = {name: ofunctions.allObjectiveFunctionWeka(list(range(X.shape[1])), X, y, type_of=of)
                  for name, (X, y) in FILES.items()}
        whole_set.append({'OF': of, 'Table 3': TABLE3_WHOLE_SET[of], **values})
    for of in SEPARABLE_OBJECTIVES + ['relieff']:
        for name, (X, y) in FILES.items():
            rankings[name, of] = rank_features(X, y, of)
finally:
    jvm.stop()


Nov 27, 2026 1:27:59 AM com.github.fommil.netlib.ARPACK <clinit>
Nov 27, 2026 1:27:59 AM com.github.fommil.netlib.ARPACK <clinit>


### 5.1 Merits of the subsets of the three rounds

In [8]:
merits = pd.DataFrame(merits)
for name in FILES:
    merits[f'reproduced on {name}'] = [same(a, b) for a, b in zip(merits['logged by the reducer'], merits[name])]
display(merits[merits['round'] == 3])
display(merits.groupby('round')[[f'reproduced on {name}' for name in FILES]].sum())

statement(f'All {len(merits)} merits logged by the reducer (3 rounds) are reproduced on {FULL_FILE}',
          merits[f'reproduced on {FULL_FILE}'].all())
statement(f'None of them is reproduced on {TRAIN_FILE}', not merits[f'reproduced on {TRAIN_FILE}'].any())

,round,Meta,OF,L,logged by the reducer,train_normal_rtf.csv,normal_rtf.csv,reproduced on train_normal_rtf.csv,reproduced on normal_rtf.csv
36,3,DE,chi2,10,37.181303,35.447290,37.181303,False,True
37,3,DE,gain_ratio,10,0.103818,0.109775,0.103818,False,True
38,3,DE,information_gain,10,0.071213,0.077334,0.071213,False,True
39,3,DE,mi,10,0.082621,0.084706,0.082621,False,True
40,3,DE,relieff,18,0.014495,0.013992,0.014495,False,True
41,3,DE,sim_uncertainty,10,0.078548,0.083722,0.078548,False,True
42,3,GA,chi2,10,34.505093,33.142945,34.505093,False,True
43,3,GA,gain_ratio,10,0.090621,0.097563,0.090621,False,True
44,3,GA,information_gain,10,0.064532,0.068563,0.064532,False,True
45,3,GA,mi,10,0.078270,0.077305,0.078270,False,True


,reproduced on train_normal_rtf.csv,reproduced on normal_rtf.csv
round,,
1,0,18
2,0,18
3,0,18


HOLDS          All 54 merits logged by the reducer (3 rounds) are reproduced on normal_rtf.csv
HOLDS          None of them is reproduced on train_normal_rtf.csv


### 5.2 Merit of the 70 genes (Table 3, column "Whole set")

Table 3 prints 3 decimals, so this check is weaker than 5.1: the cell shows, for each objective, the value on each
file rounded to 3 decimals and which file is closer to the printed value.

In [9]:
whole_set = pd.DataFrame(whole_set)
for name in FILES:
    whole_set[f'{name} rounded'] = whole_set[name].round(3)
whole_set['closer to Table 3'] = [TRAIN_FILE if abs(t - p) < abs(f - p) else FULL_FILE
                                  for p, t, f in zip(whole_set['Table 3'], whole_set[TRAIN_FILE], whole_set[FULL_FILE])]
display(whole_set)
statement(f'For every objective, the 70-gene merit on {FULL_FILE} is closer to Table 3 than the one on {TRAIN_FILE}',
          (whole_set['closer to Table 3'] == FULL_FILE).all())
differ = whole_set[whole_set[f'{FULL_FILE} rounded'] != whole_set['Table 3']]
print(f'Rounded to 3 decimals, the values on {FULL_FILE} are those printed in Table 3 except for: '
      + (', '.join(f"{r.OF} ({r[FULL_FILE]:.6f}, printed {r['Table 3']})" for _, r in differ.iterrows()) or 'none'))

,OF,Table 3,train_normal_rtf.csv,normal_rtf.csv,train_normal_rtf.csv rounded,normal_rtf.csv rounded,closer to Table 3
0,mi,0.034,0.036790,0.033766,0.037,0.034,normal_rtf.csv
1,sim_uncertainty,0.031,0.034072,0.030764,0.034,0.031,normal_rtf.csv
2,gain_ratio,0.037,0.039389,0.036585,0.039,0.037,normal_rtf.csv
3,relieff,0.010,0.009541,0.009897,0.010,0.010,normal_rtf.csv
4,chi2,14.573,14.828242,14.572457,14.828,14.572,normal_rtf.csv


HOLDS          For every objective, the 70-gene merit on normal_rtf.csv is closer to Table 3 than the one on train_normal_rtf.csv
Rounded to 3 decimals, the values on normal_rtf.csv are those printed in Table 3 except for: chi2 (14.572457, printed 14.573)


### 5.3 Top-10 ranking against the DE subsets

For the separable objectives, the top 10 of a file is the maximum of Eq. (11) on that file. A DE subset equal to the
top 10 of one file and not of the other shows which file DE searched. aReliefF is not separable (a gene's weight
depends on the other genes), so it is shown for information only.

In [10]:
de = reducer_rounds[3].set_index('OF').query("Meta == 'DE'")['Subset'].map(sorted)
top10 = []
for of in SEPARABLE_OBJECTIVES + ['relieff']:
    row = {'OF': of, 'DE subset (round 3)': de[of]}
    for name in FILES:
        ranking, _ = rankings[name, of]
        top = sorted(int(i) for i in ranking[:K])
        row[f'top 10 of {name}'] = top
        row[f'DE = top 10 of {name}'] = top == de[of]
    # Genes DE kept that the training data alone would not have ranked in the top 10, as g numbers
    train_top = set(row[f'top 10 of {TRAIN_FILE}'])
    row['in DE only (g)'] = [f'g{i + 1}' for i in de[of] if i not in train_top]
    row[f'in top 10 of {TRAIN_FILE} only (g)'] = [f'g{i + 1}' for i in sorted(train_top - set(de[of]))]
    top10.append(row)
top10 = pd.DataFrame(top10).set_index('OF')
display(top10)

separable = top10.loc[SEPARABLE_OBJECTIVES]
statement(f'For the {len(SEPARABLE_OBJECTIVES)} separable objectives, every DE subset is the top 10 of {FULL_FILE}',
          separable[f'DE = top 10 of {FULL_FILE}'].all())
print(f"DE subsets equal to the top 10 of {TRAIN_FILE}: "
      f"{int(separable[f'DE = top 10 of {TRAIN_FILE}'].sum())} of {len(separable)} "
      f"({', '.join(separable.index[separable[f'DE = top 10 of {TRAIN_FILE}']]) or 'none'})")

,DE subset (round 3),top 10 of train_normal_rtf.csv,DE = top 10 of train_normal_rtf.csv,top 10 of normal_rtf.csv,DE = top 10 of normal_rtf.csv,in DE only (g),in top 10 of train_normal_rtf.csv only (g)
OF,,,,,,,
mi,"[3, 9, 10, 50, 52, 57, 59, 63, 65, 67]","[9, 10, 21, 23, 35, 50, 52, 59, 63, 65]",False,"[3, 9, 10, 50, 52, 57, 59, 63, 65, 67]",True,"[g4, g58, g68]","[g22, g24, g36]"
sim_uncertainty,"[9, 11, 14, 21, 42, 44, 50, 58, 63, 65]","[9, 14, 21, 32, 34, 42, 50, 63, 65, 67]",False,"[9, 11, 14, 21, 42, 44, 50, 58, 63, 65]",True,"[g12, g45, g59]","[g33, g35, g68]"
gain_ratio,"[14, 21, 33, 34, 42, 44, 50, 58, 63, 65]","[14, 21, 33, 34, 42, 44, 50, 58, 63, 65]",True,"[14, 21, 33, 34, 42, 44, 50, 58, 63, 65]",True,[],[]
chi2,"[11, 14, 38, 49, 50, 52, 58, 63, 65, 67]","[9, 11, 12, 14, 32, 50, 52, 63, 65, 67]",False,"[11, 14, 38, 49, 50, 52, 58, 63, 65, 67]",True,"[g39, g50, g59]","[g10, g13, g33]"
information_gain,"[9, 11, 14, 49, 50, 52, 58, 63, 65, 67]","[9, 11, 12, 14, 32, 50, 52, 63, 65, 67]",False,"[9, 11, 14, 49, 50, 52, 58, 63, 65, 67]",True,"[g50, g59]","[g13, g33]"
relieff,"[2, 6, 9, 16, 18, 20, 22, 23, 24, 28, 32, 37, 38, 48, 49, 50, 57, 62]","[2, 9, 23, 24, 28, 32, 37, 50, 57, 59]",False,"[2, 20, 23, 24, 28, 32, 37, 48, 57, 59]",False,"[g7, g17, g19, g21, g23, g39, g49, g50, g63]",[g60]


HOLDS          For the 5 separable objectives, every DE subset is the top 10 of normal_rtf.csv
DE subsets equal to the top 10 of train_normal_rtf.csv: 1 of 5 (gain_ratio)


## 6. The χ² merits of Table 2

Table 2 prints the aCHI2 merits on a 0–1 scale (DE 1.000, GA 0.879, SA 0.778 in round 3) and Table 3 prints the
raw Eq. (11) values (37.181, 34.505, 32.255). `processDataset.ipynb` (September 2023) holds the normalization:
a hand-typed list of χ² merits, scaled with `MinMaxScaler`.

In [11]:
notebook = json.loads(Path(NORMALIZATION_NOTEBOOK).read_text(encoding='utf-8'))
sources = [''.join(cell['source']) for cell in notebook['cells'] if cell['cell_type'] == 'code']
listed = next(s for s in sources if s.lstrip().startswith('array = ['))
values = np.array(ast.literal_eval(listed.split('=', 1)[1].strip()))
low, high = values.min(), values.max()
print('Cells of processDataset.ipynb that scale the list:')
for s in sources:
    if 'MinMaxScaler' in s or 'min(a)' in s:
        print('   ', s.replace('\n', '\n    '))
print(f'\nThe list has {len(values)} χ² merits; min = {low}, max = {high}')

chi2 = []
for r in (1, 2, 3):
    reducer = read_results(REDUCER_OUTPUTS[r]).query("OF == 'chi2'").set_index('Meta')
    submitted = read_results(SUBMITTED[r]).query("OF == 'chi2'").set_index('Meta')
    for meta, row in reducer.iterrows():
        normalized = (row['Merit'] - low) / (high - low)
        chi2.append({'round': r, 'Meta': meta, 'L': row['Length'], 'raw (reducer)': row['Merit'],
                     'raw value in the list': bool(np.isclose(values, row['Merit'], rtol=0, atol=TOLERANCE).any()),
                     '(raw - min) / (max - min)': normalized, 'Table 2 (spreadsheet)': submitted.loc[meta, 'Merit'],
                     'same': same(normalized, submitted.loc[meta, 'Merit'])})
chi2 = pd.DataFrame(chi2)
display(chi2)
statement('Every χ² merit of Table 2 is (raw - min) / (max - min), with the min and max of the list in processDataset.ipynb',
          chi2['same'].all())
round3_chi2 = chi2[chi2['round'] == 3].set_index('Meta')['raw (reducer)']
statement('The χ² merits of Table 3 are the raw values logged by the reducer',
          all(same(round(round3_chi2[meta], 3), printed, 1e-9) for meta, printed in TABLE3_CHI2.items()))

Cells of processDataset.ipynb that scale the list:
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import MinMaxScaler
    min(a)
    scaler = MinMaxScaler()
    scaler.fit(a)
    procesed = scaler.transform(a)
    scaler = MinMaxScaler()
    scaler.fit(data)
    procesed = scaler.transform(data)
    procesed

The list has 90 χ² merits; min = 14.998303, max = 37.1813027


,round,Meta,L,raw (reducer),raw value in the list,(raw - min) / (max - min),Table 2 (spreadsheet),same
0,1,DE,10,37.181303,True,1.000000,1.000000,True
1,1,GA,32,21.584387,True,0.296898,0.296898,True
2,1,SA,29,21.366761,True,0.287087,0.287087,True
3,2,DE,10,37.181303,True,1.000000,1.000000,True
4,2,GA,14,30.622795,True,0.704345,0.704345,True
5,2,SA,12,28.436945,True,0.605808,0.605808,True
6,3,DE,10,37.181303,True,1.000000,1.000000,True
7,3,GA,10,34.505093,True,0.879358,0.879358,True
8,3,SA,10,32.255296,True,0.777938,0.777938,True


HOLDS          Every χ² merit of Table 2 is (raw - min) / (max - min), with the min and max of the list in processDataset.ipynb
HOLDS          The χ² merits of Table 3 are the raw values logged by the reducer


## 7. Summary

In [12]:
display(pd.DataFrame(CHECKS))

,Statement,Holds
0,"normal_rtf.csv is exactly the rows of train_normal_rtf.csv plus those of test_normal_rtf.csv, and no row is in both",True
1,The 30 real test patients (test_rtf_v2_cleaned_normal.csv) are among the 40 rows of test_normal_rtf.csv,True
2,The 30 real test patients are rows of normal_rtf.csv,True
3,"In the three copies, every menu option that runs the multi-round reducer loads normal_rtf.csv (options 6 and 9 in pi...",True
4,train_normal_rtf.csv is loaded only by menu options that do not run metaheuristics,True
5,"The reducer output files hold the same subsets as the manuscript spreadsheets, in all three rounds",True
6,"Their merits are also the same, except for χ², which the spreadsheets hold normalized (section 6)",True
7,All 54 merits logged by the reducer (3 rounds) are reproduced on normal_rtf.csv,True
8,None of them is reproduced on train_normal_rtf.csv,True
9,"For every objective, the 70-gene merit on normal_rtf.csv is closer to Table 3 than the one on train_normal_rtf.csv",True
